In [ ]:
#| hide
# from nbdev_mcp import *  # noqa

# nbdev_mcp

> nbdev MCP server (multi-project, rich output, workflow-aware) — Python 3.12+

### Features

- Works from anywhere: select a project by path or alias, or pass project= per call.
  
- Multi-project support: bookmarks (alias -> path), discovery via `NBDEV_PROJECTS` or common roots.
  
- **Core nbdev tools**: version-aware command handling for v2 (`nbdev_prepare`, `nbdev_export`, `nbdev_test`) and v3 (`nbdev-prepare`, `nbdev-export`, `nbdev-test`), plus `pytest`, `ensure_env`, `export_env` (mamba/conda).
  
- **Notebook editing**: `check_if_generated`, `find_source_notebook`, edit/read/add notebook cells.
  
- **Linting**: `validate_inits`, `lint_rules` (duplicate exports, relative imports, `__all__` detection), `lint_main_guards`.
  
- **Analysis**: `modidx_audit`, `find_symbol`, `dependency_tree`, `dependency_snapshot`, `analyze_dependency_order`.
  
- **Testing**: `run_tutorials`, `scan_notebook_errors`, `generate_pytests`, `scaffold_test_utils`.
  
- **Code generation**: `generate_stubs`, `aggregate_todos`, `aggregate_bugs`.
  
- **Workflow guidance**: Prompts (including `reuse_first_checklist`) that instruct LLMs to use nbdev properly.
  
- **Resources**: project summary, tree, detected nbdev settings file (`settings.ini`/`settings.toml`/`pyproject.toml`), env file, roadmap, learning links (safe, read-only).
  
- **Transports**: stdio (for desktop clients), streamable-http (built-in default HTTP), or HTTP via Uvicorn.
  
- Rich-formatted "pretty" output for UI display; never prints to stdout in stdio mode.


### Philosophy

In nbdev, `.ipynb` notebooks are _source code_. Python `.py` files are _generated_.

This MCP guides LLMs to follow this philosophy by providing tools to:

1. Check if files are generated (`check_if_generated`)
2. Find source notebooks (`find_source_notebook`)
3. Edit notebooks instead of `.py` files (`edit_notebook_cell`)
4. Always run the project-appropriate export command after editing (`nbdev_export` for v2 or `nbdev-export` for v3)
5. Search for existing symbols before adding new code (`find_symbol`)
6. Analyze dependencies and avoid duplication (`dependency_tree`, `dependency_snapshot`)
7. Lint for best practices (`lint_rules`, `validate_inits`)


### Conventions

- `nbs/index.ipynb` becomes README.md and does not need `default_exp`.
- Avoid private names (leading underscore) for functions/classes/vars; dunder methods like `__init__` are OK.
- Dead-code reports are signals: some exports are used in tutorials/docs, but unused exports can indicate duplication (e.g., A vs A2).
- Keep living docs current: `ROADMAP.md`, `TODOs.md`, `*_PLAN.md`, and agent docs under `.claude/` or `.codex/`.
- All scripting/CLI logic must live in `nbs/` and be exposed via detected nbdev settings (`settings.ini`, `settings.toml`, or `pyproject.toml` `[tool.nbdev]`).
- Repo-level `.md` files can be added to `nbs/index.ipynb` as markdown cells (use `split_markdown_cells`).
- Use `nbdev://repo-markdown` to enumerate repo/agent markdown docs and `nbdev://repo-markdown/{doc_key}` to read one safely.


## Integrations

### VS Code

#### Command Dialog (<kbd>Cmd</kbd> + <kbd>Shift</kbd> + <kbd>P</kbd>)

Press <kbd>Cmd</kbd> + <kbd>Shift</kbd> + <kbd>P</kbd> to open up the command dialog in VS Code. Then start typing `'MCP'` to find MCP related commands.

You are looking for `'MCP: Add Server...'`

#### Config File

Use the built-in installer (recommended):

```sh
python -m nbdev_mcp install vscode --auto-start
python -m nbdev_mcp update vscode --strategy merge --dry-run
python -m nbdev_mcp status
```

Notes:

- VS Code MCP config is `mcp.json` with server key `servers`.
- `--auto-start` also enables `chat.mcp.autostart` in `settings.json`.

Manual path reference:

For macOS, VS Code user config is typically under:

```sh
~/Library/Application Support/Code/User/
```

The MCP config file is `mcp.json`.

```sh
$ touch ~/Library/Application Support/Code/User/mcp.json
```

Then in `mcp.json`:

```json
# ~/Library/Application Support/Code/User/mcp.json
{
    "servers": {
        // ... other servers
        "nbdev": {
            "type": "stdio",
            "command": "python",
            "args": ["-u", "-m", "nbdev_mcp"]
        }
    }
}
```

### Cursor

#### Config File

Use the built-in installer (recommended):

```sh
python -m nbdev_mcp install cursor --auto-start
python -m nbdev_mcp update cursor --strategy merge --dry-run
python -m nbdev_mcp status
```

Notes:

- Cursor MCP config is `mcp.json` with server key `servers`.
- `--auto-start` also enables `chat.mcp.autostart` in `settings.json`.

Manual path reference (macOS):

```sh
~/Library/Application Support/Cursor/User/
```

The MCP config file is `mcp.json`.


### Claude Code

#### Config File

Use the built-in installer (recommended):

```sh
python -m nbdev_mcp install claude
python -m nbdev_mcp update claude --strategy merge --dry-run
python -m nbdev_mcp status
```

Supported Claude config variants:

1. Claude Code: `~/.claude.json`
2. Claude Desktop:
   - macOS: `~/Library/Application Support/Claude/claude_desktop_config.json`
   - Linux: `~/.config/Claude/claude_desktop_config.json`
   - Windows: `%APPDATA%/Claude/claude_desktop_config.json`

Example (`~/.claude.json`):

```json
{
  "mcpServers": {
    "nbdev": {
      "command": "/Path/to/python",
      "args": ["-u", "-m", "nbdev_mcp"]
    }
  }
}
```


#### Command Line

```sh
PYTHON="python3" # or /Path/to/{mamba|conda|micromamba}/envs/{env_name}/bin/python

claude mcp remove nbdev 2>/dev/null || true
claude mcp add --scope user --transport stdio nbdev -- \
    "$PYTHON" -u -m nbdev_mcp
```


### Codex

#### Config File

Use the built-in installer (recommended):

```sh
python -m nbdev_mcp install codex
python -m nbdev_mcp update codex --strategy merge --dry-run
python -m nbdev_mcp status
```

Codex config variants supported by the installer:

1. Preferred: `~/.codex/config.toml`
2. Legacy: `~/.codex/config.json`

TOML example:

```toml
# https://developers.openai.com/codex/local-config/
model = "gpt-5.1-codex-max"
model_provider = "openai"

[mcp_servers.nbdev]
command = "/Path/to/python"
args = ["-u", "-m", "nbdev_mcp"]
```

Legacy JSON example:

```json
{
  "mcpServers": {
    "nbdev": {
      "command": "/Path/to/python",
      "args": ["-u", "-m", "nbdev_mcp"]
    }
  }
}
```


#### Command Line

```sh
codex mcp add mcp-nbdev \
    --env NBDEV_PROJECTS="~/dev:~/Projects" \
    /Path/to/{mamba|conda|micromamba}/envs/{env_name}/bin/python \
    -u -m nbdev_mcp
```


## Developer Guide

If you are new to using `nbdev` here are some useful pointers to get you started.

### Install nbdev_mcp in Development Mode


```sh
# make sure nbdev_mcp is installed in development mode
$ pip install -e .

# make changes under nbs/ directory
# ...

# compile so changes apply to nbdev_mcp
# nbdev v2:
$ nbdev_prepare
# nbdev v3:
$ nbdev-prepare
```


## Usage

### Installation

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/s01st/nbdev_mcp.git
```

or from [conda][conda]

```sh
$ conda install -c s01st nbdev_mcp
```

or from [pypi][pypi]


```sh
$ pip install nbdev_mcp
```


[repo]: https://github.com/s01st/nbdev_mcp
[docs]: https://s01st.github.io/nbdev_mcp/
[pypi]: https://pypi.org/project/nbdev_mcp/
[conda]: https://anaconda.org/s01st/nbdev_mcp

### Documentation

Documentation is hosted on [GitHub Pages](https://s01st.github.io/nbdev_mcp/). Repository and package info is available on [GitHub](https://github.com/s01st/nbdev_mcp), [PyPI](https://pypi.org/project/nbdev_mcp/), and [Anaconda](https://anaconda.org/s01st/nbdev_mcp).

## How to use

```sh
$ python -m nbdev_mcp --transport http --host 127.0.0.1 --port 8766 --path /mcp
```

```sh
(core) ➜  ~ python -m nbdev_mcp --transport http --host 127.0.0.1 --port 8766 --path /mcp
INFO:     Started server process [47867]
INFO:     Waiting for application startup.
[12/03/25 11:06:02] INFO     StreamableHTTP session manager started   streamable_http_manager.py:110
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8766 (Press CTRL+C to quit)
```